In [ ]:
from pathlib import Path
import os
import gc
import random
import logging
from datetime import datetime
from collections import defaultdict

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as T
import torchvision.transforms.functional as TF
from torchvision.transforms import InterpolationMode
from PIL import Image
from tqdm import tqdm

from model import SpatioTemporalDynamicClassifier


def setup_logging(log_dir="logs"):
    os.makedirs(log_dir, exist_ok=True)
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    log_file = os.path.join(log_dir, f"train_{timestamp}.log")

    for handler in logging.root.handlers[:]:
        logging.root.removeHandler(handler)

    logging.basicConfig(
        level=logging.INFO,
        format='%(asctime)s - %(message)s',
        handlers=[
            logging.StreamHandler(),
            logging.FileHandler(log_file, encoding='utf-8')
        ]
    )
    logger = logging.getLogger(__name__)
    logger.info(f"✅ 日志模块启动成功！训练记录将保存至: {log_file}")
    return logger


def clear_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


MVITAC_SOFT_CLASSES = {7, 8, 9, 11, 13}


ROUGH_CLASSES = {0, 5, 12, 14, 15, 16, 17}
SMOOTH_CLASSES = {1, 2, 4, 6, 11}


def build_optimizer(model, lr_backbone, lr_head, weight_decay):
    backbone_decay, backbone_no_decay = [], []
    head_decay, head_no_decay = [], []

    def is_backbone(name: str):
        return (name.startswith("vision_encoder.") or name.startswith("tactile_encoder.") or
                name.startswith("v_spatial_proj.") or name.startswith("t_proj."))

    def is_no_decay(name: str, p: torch.nn.Parameter):
        if name.endswith(".bias") or p.ndim <= 1: return True
        if "norm" in name.lower() or "pos" in name.lower(): return True
        return False

    for name, p in model.named_parameters():
        if not p.requires_grad: continue
        if is_backbone(name):
            (backbone_no_decay if is_no_decay(name, p) else backbone_decay).append(p)
        else:
            (head_no_decay if is_no_decay(name, p) else head_decay).append(p)

    param_groups = [
        {"params": backbone_decay, "lr": lr_backbone, "weight_decay": weight_decay},
        {"params": backbone_no_decay, "lr": lr_backbone, "weight_decay": 0.0},
        {"params": head_decay, "lr": lr_head, "weight_decay": weight_decay},
        {"params": head_no_decay, "lr": lr_head, "weight_decay": 0.0},
    ]
    return torch.optim.AdamW([g for g in param_groups if len(g["params"]) > 0], betas=(0.9, 0.98), eps=1e-6)


class PairedRandomResizedCropFlipSequence:
    def __init__(self, size=224, scale=(0.8, 1.0), ratio=(3/4, 4/3), hflip_p=0.5):
        self.size = (size, size) if isinstance(size, int) else size
        self.scale = scale
        self.ratio = ratio
        self.hflip_p = hflip_p

    def __call__(self, rgb_imgs, tac_imgs):
        i, j, h, w = T.RandomResizedCrop.get_params(rgb_imgs[0], scale=self.scale, ratio=self.ratio)
        do_flip = (random.random() < self.hflip_p)
        out_rgb, out_tac = [], []
        for r, t in zip(rgb_imgs, tac_imgs):
            r = TF.resized_crop(r, i, j, h, w, self.size, interpolation=InterpolationMode.BILINEAR)
            t = TF.resized_crop(t, i, j, h, w, self.size, interpolation=InterpolationMode.BILINEAR)
            if do_flip:
                r, t = TF.hflip(r), TF.hflip(t)
            out_rgb.append(r)
            out_tac.append(t)
        return out_rgb, out_tac


class TagSequenceDataset(Dataset):
    def __init__(self, root, mode="train", seq_txt_name=None, paired_transform_seq=None, transform_rgb=None, transform_tactile=None):
        self.root = Path(root)
        self.mode = mode
        self.paired_transform_seq = paired_transform_seq
        self.transform_rgb = transform_rgb
        self.transform_tactile = transform_tactile

        data_file = self.root / (seq_txt_name or f"{mode}_seq.txt")
        with open(data_file, "r", encoding="utf-8") as f:
            raw_lines = [line.strip() for line in f.readlines() if line.strip()]

        self.samples = []
        invalid = 0

        for line in raw_lines:
            try:
                seq_part, target = line.rsplit(",", 1)
                target = int(target)
                if target == -1: continue

                frame_paths = seq_part.split()
                rgb_paths, tac_paths = [], []
                for raw in frame_paths:
                    p = Path(raw)
                    dir_path = self.root / p.parts[0]
                    rgb_p = dir_path / "video_frame" / p.name
                    tac_p = dir_path / "gelsight_frame" / p.name
                    if not (rgb_p.exists() and tac_p.exists()): raise FileNotFoundError()
                    rgb_paths.append(rgb_p)
                    tac_paths.append(tac_p)

                self.samples.append((rgb_paths, tac_paths, target))
            except Exception:
                invalid += 1

        print(f"[TAG-SEQ-{mode}] 有效样本 {len(self.samples)} | 跳过无效 {invalid}")

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        rgb_paths, tac_paths, target = self.samples[idx]
        rgb_imgs = [Image.open(p).convert("RGB") for p in rgb_paths]
        tac_imgs = [Image.open(p).convert("RGB") for p in tac_paths]

        if self.paired_transform_seq:
            rgb_imgs, tac_imgs = self.paired_transform_seq(rgb_imgs, tac_imgs)

        rgb_seq = [self.transform_rgb(img) if self.transform_rgb else T.ToTensor()(img) for img in rgb_imgs]
        tac_seq = [self.transform_tactile(img) if self.transform_tactile else T.ToTensor()(img) for img in tac_imgs]

        return torch.stack(rgb_seq, dim=0), torch.stack(tac_seq, dim=0), target


@torch.no_grad()
def evaluate(model, loader, device, criterion, use_amp=True):
    model.eval()
    total_loss, total_correct, total_count = 0.0, 0, 0

    hs_correct, hs_total = 0, 0
    rs_correct, rs_total = 0, 0

    for rgb, tac, label in loader:
        rgb, tac, label = rgb.to(device, non_blocking=True), tac.to(device, non_blocking=True), label.to(device, non_blocking=True)

        with torch.cuda.amp.autocast(enabled=(use_amp and device.type == "cuda")):
            logits = model(rgb, tac)
            loss = criterion(logits, label)

        preds = logits.argmax(dim=1)
        total_loss += loss.item() * label.size(0)
        total_correct += (preds == label).sum().item()
        total_count += label.size(0)

        for p, l in zip(preds.tolist(), label.tolist()):

            hs_total += 1

            is_l_soft = (l in MVITAC_SOFT_CLASSES)
            is_p_soft = (p in MVITAC_SOFT_CLASSES)
            if is_l_soft == is_p_soft:
                hs_correct += 1

            if l in ROUGH_CLASSES or l in SMOOTH_CLASSES:
                rs_total += 1
                is_l_rough = l in ROUGH_CLASSES
                is_p_rough = p in ROUGH_CLASSES
                if (is_l_rough and is_p_rough) or (not is_l_rough and p in SMOOTH_CLASSES):
                    rs_correct += 1

    avg_loss = total_loss / max(1, total_count)
    avg_acc = 100.0 * total_correct / max(1, total_count)
    hs_acc = (100.0 * hs_correct / hs_total) if hs_total > 0 else 0.0
    rs_acc = (100.0 * rs_correct / rs_total) if rs_total > 0 else 0.0

    return {'loss': avg_loss, 'acc': avg_acc, 'hs_acc': hs_acc, 'rs_acc': rs_acc}


def load_mambavision_pretrained(model, ckpt_path, verbose=True):
    if not os.path.exists(ckpt_path): raise FileNotFoundError()
    ckpt = torch.load(ckpt_path, map_location="cpu")
    state = ckpt.get("state_dict", ckpt.get("model", ckpt))
    model_state = model.state_dict()
    loaded = []

    for k, v in state.items():
        if "head" in k or "classifier" in k: continue
        target_key_v = f"vision_encoder.model.{k}"
        target_key_t = f"tactile_encoder.model.{k}"

        if target_key_v in model_state and model_state[target_key_v].shape == v.shape:
            model_state[target_key_v].copy_(v)
            loaded.append(target_key_v)
        if target_key_t in model_state and model_state[target_key_t].shape == v.shape:
            model_state[target_key_t].copy_(v)
            loaded.append(target_key_t)

    if verbose: print(f"\n[Pretrain] Loaded {len(loaded)} params\n")
    return loaded


def main():
    logger = setup_logging()

    config = {
        "data_folder": "",
        "num_classes": 20,
        "epochs": 30,
        "batch_size": 8,
        "num_workers": 8,
        "grad_clip": 1.0,

        "lr_backbone": 3e-6,
        "lr_head": 3e-5,
        "weight_decay": 1e-2,
        "freeze_backbone": False,
        "dropout": 0.3,
        "label_smoothing": 0.1,

        "pretrained_path": "mambavision_tiny_1k.pth.tar",
        "d_model": 256,
        "d_state": 16,
        "hierarchical": True,
    }

    logger.info("="*60)
    logger.info(" 时空双流网络训练 ")
    for k, v in config.items(): logger.info(f"  {k}: {v}")

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    paired_geom_seq = PairedRandomResizedCropFlipSequence(size=224, scale=(0.8, 1.0), hflip_p=0.5)
    train_transform_rgb = T.Compose([T.ColorJitter(0.2, 0.2, 0.2, 0.05), T.ToTensor(), T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])
    tactile_train_transform = T.Compose([T.ToTensor(), T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])
    test_transform_rgb = T.Compose([T.Resize(256), T.CenterCrop(224), T.ToTensor(), T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])
    tactile_test_transform = T.Compose([T.Resize(256), T.CenterCrop(224), T.ToTensor(), T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])

    logger.info("Loading datasets...")
    train_dataset = TagSequenceDataset(config["data_folder"], "train", "train.txt", paired_geom_seq, train_transform_rgb, tactile_train_transform)
    test_dataset = TagSequenceDataset(config["data_folder"], "test", "test.txt", None, test_transform_rgb, tactile_test_transform)

    train_loader = DataLoader(train_dataset, batch_size=config["batch_size"], shuffle=True, num_workers=config["num_workers"], drop_last=True, pin_memory=True)
    test_loader = DataLoader(test_dataset, batch_size=config["batch_size"], shuffle=False, num_workers=config["num_workers"], pin_memory=True)

    logger.info("Building Model...")
    model = SpatioTemporalDynamicClassifier(
        num_classes=config["num_classes"],
        d_model=config["d_model"],
        d_state=config["d_state"],
        dropout=config["dropout"],
        hierarchical=config["hierarchical"],
    ).to(device)

    if config["pretrained_path"] and os.path.exists(config["pretrained_path"]):
        load_mambavision_pretrained(model, ckpt_path=config["pretrained_path"], verbose=False)

    optimizer = build_optimizer(model, config["lr_backbone"], config["lr_head"], config["weight_decay"])
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=config["epochs"], eta_min=1e-7)
    criterion = nn.CrossEntropyLoss(label_smoothing=config["label_smoothing"]).to(device)
    scaler = torch.cuda.amp.GradScaler(enabled=True)

    best_acc = 0.0
    os.makedirs("checkpoints", exist_ok=True)
    logger.info("开始训练...")

    for epoch in range(config["epochs"]):
        model.train()
        clear_memory()
        total_loss, total_correct, total_count = 0.0, 0, 0

        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{config['epochs']}")
        for rgb, tac, label in pbar:
            rgb, tac, label = rgb.to(device, non_blocking=True), tac.to(device, non_blocking=True), label.to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)
            with torch.cuda.amp.autocast(enabled=True):
                logits = model(rgb, tac)
                loss = criterion(logits, label)

            scaler.scale(loss).backward()
            if config["grad_clip"] > 0:
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), config["grad_clip"])

            scaler.step(optimizer)
            scaler.update()

            pred = logits.argmax(dim=1)
            total_correct += (pred == label).sum().item()
            total_count += label.size(0)
            total_loss += loss.item() * label.size(0)
            pbar.set_postfix({"loss": f"{loss.item():.4f}", "acc": f"{100.0 * (pred == label).float().mean().item():.2f}%"})

        scheduler.step()
        train_acc = 100.0 * total_correct / max(1, total_count)

        test_results = evaluate(model, test_loader, device, criterion, use_amp=True)
        test_acc = test_results['acc']

        epoch_log = (
            f"[Epoch {epoch+1}/{config['epochs']}] "
            f"Train Acc: {train_acc:.2f}% | "
            f"Test Acc (总20类): {test_acc:.2f}% | "
            f"Hard/Soft (对齐MViTac): {test_results['hs_acc']:.2f}% | "
            f"Rough/Smooth: {test_results['rs_acc']:.2f}%"
        )
        logger.info(epoch_log)

        if test_acc > best_acc:
            best_acc = test_acc
            save_path = os.path.join("checkpoints", f"best_model_hierarchical_{config['hierarchical']}.pth")
            torch.save(model.state_dict(), save_path)
            logger.info(f" 新最佳模型！已保存至 {save_path}")

    logger.info(f" 训练完成! 历史最高 20分类准确率: {best_acc:.2f}%")

if __name__ == "__main__":
    main()
